# 02 — Exploratory Analysis

**Business purpose.** Separate changes in signed recorded value from eligible positive line sales and transaction volume; then inspect product/country contribution. All currency values are GBP recorded transaction-line values, not profit.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from src.config import DEFAULT_CLEANED, MARTS_DIR

if not DEFAULT_CLEANED.exists():
    raise FileNotFoundError("Run ingestion, audit, and `python -m src.clean` first.")
lines = pd.read_csv(DEFAULT_CLEANED, low_memory=False, parse_dates=["invoice_date"], dtype={"invoice_no":"string", "stock_code":"string", "customer_id":"string", "country":"string"})
monthly = pd.read_csv(MARTS_DIR / "monthly_sales.csv", parse_dates=["month"])
print("Fact rows:", f"{len(lines):,}")
print("Date coverage:", lines["invoice_date"].min(), "to", lines["invoice_date"].max())
monthly.tail()

## Monthly performance

Compare net recorded line value, gross positive line sales, and eligible invoice count. Keep the measures separate; do not attribute changes to demand, pricing, or seasonality without evidence.

In [ ]:
fig, ax = plt.subplots(figsize=(12, 5))
ax.plot(monthly["month"], monthly["gross_positive_line_sales_gbp"], label="Gross positive line sales (GBP)")
ax.plot(monthly["month"], monthly["net_recorded_sales_gbp"], label="Net recorded line value (GBP)")
ax.set_title("Recorded sales measures by month")
ax.set_ylabel("GBP")
ax.set_xlabel("Month")
ax.legend()
ax.grid(True, alpha=0.25)
fig.autofmt_xdate()
plt.show()

fig, ax = plt.subplots(figsize=(12, 4))
ax.bar(monthly["month"], monthly["sales_invoices"], width=20)
ax.set_title("Eligible sales invoices by month")
ax.set_ylabel("Distinct invoices")
ax.set_xlabel("Month")
fig.autofmt_xdate()
plt.show()

## Product and country contribution

Contribution percentages use the sum of eligible positive sales for rows with an available product/country label. Missing labels are not deleted from the fact table; the attribution denominator may therefore differ from total eligible sales.

In [ ]:
products = pd.read_csv(MARTS_DIR / "product_performance.csv")
countries = pd.read_csv(MARTS_DIR / "country_performance.csv")
display(products.head(15))
display(countries.head(15))

fig, ax = plt.subplots(figsize=(10, 6))
top = products.head(10).sort_values("gross_positive_line_sales_gbp")
ax.barh(top["stock_code"].astype(str), top["gross_positive_line_sales_gbp"])
ax.set_title("Top 10 products by eligible positive line sales")
ax.set_xlabel("GBP")
ax.set_ylabel("Stock code")
plt.show()

## Interpretation checkpoint

For your portfolio notes, identify one observed change in sales, compare it with invoice volume, and write one alternative explanation. Then cite the exact month(s), values, query, and definition. If the data do not support a clear pattern, say so.